# Lab Solutions: APIs

##### Instructions: 

Write your own Python script to answer the following questions: 
1. Which of these embassies is closest to the White House in meters? What is the address? 
2. If I wanted to hold a morning meeting there, which cafe would you suggest (best rating and closest)?
3. If I wanted to hold an upscale evening meeting there, which fancy bar would you suggest? 

Hint: 
- You will need to enable the `Google Places API`.
- You may find this page useful to learn about different findinging nearby places https://www.geeksforgeeks.org/python-fetch-nearest-hospital-locations-using-googlemaps-api/

In [37]:
import importlib
import os
import googlemaps
from getpass import getpass

In [38]:
os.getcwd()

'/Users/almavelazquez/Documents/GitHub/PythonCamp2024/Day05/Lecture'

In [ ]:
api_key = getpass("Enter your Google Maps API key: ")
gmaps = googlemaps.Client(key=api_key)

whitehouse = '1600 Pennsylvania Avenue, Washington, DC'

embassies = [[38.917228,-77.0522365], 
	[38.9076502, -77.0370427], 
	[38.916944, -77.048739] ]


1. Which of these embassies is closest to the White House in meters? 

In [ ]:
# save whitehouse geocode object
wh = gmaps.geocode(whitehouse)

##### Exploratory work
* Find distance to embassies

In [ ]:
test_dist = gmaps.distance_matrix(wh[0]['geometry']['location'], embassies)

In [ ]:
test_dist

In [ ]:
test_dist['rows'][0]['elements']

* Each dictionary in this list corresponds to each embassy in the list

In [ ]:
test_dist['rows'][0]['elements'][0]

In [ ]:
test_dist['rows'][0]['elements'][0]['distance']['value']

##### Put it all together in a function

In [ ]:
# define function to calculate distance to each embassy
def dist_finder(dest, origin = wh): # takes argument for destination (lat/lng), and origin with wh default
	
    # access origin latitude & longitude
    orig_lat_long = origin[0]['geometry']['location']
	
    # use distance API to calculate distance between the destination and origin lat/lngs
    dist = gmaps.distance_matrix(orig_lat_long, dest)

    # instantiate empty list
    meters=[]
    
    # for 1, 2, 3 of the embassies
    for i in range(3):
        # add the following to the empty list
        meters.append(float(dist['rows'][0]['elements'][i]['distance']['value']))
        
    return dest[meters.index(min(meters))]

# find closest embassy
closest = dist_finder(dest = embassies)

closest

What is the address? 

In [ ]:
# address of closest
closest_address = gmaps.reverse_geocode(closest)[0]['formatted_address']
closest_address

2. If I wanted to hold a morning meeting there, which cafes would you suggest for breakfast (best rating and closest)?

In [ ]:
bfast = gmaps.places_nearby(closest, type = 'cafe', 
                            rank_by = "distance", 
                            keyword = 'breakfast')['results']

In [ ]:
bfast

In [ ]:
len(bfast)

##### Closest

In [ ]:
print('Closest breakfast place is {} at {}'.format(bfast[0]['name'], bfast[0]['vicinity'])) # closest

In [ ]:
# highest rated
rating = []
for i in range(len(bfast)):
	try:
		rating.append(bfast[i]['rating'])
	except:
		rating.append(0)
best_bfast = bfast[rating.index(max(rating))]

##### Best-rated

In [ ]:
print('Best breakfast place is {} at {} with {}'.format(best_bfast['name'], 
                                                best_bfast['vicinity'], best_bfast['rating'])) # highest rated

3. If I wanted to hold an upscale evening meeting there, which fancy bar would you suggest? 

In [ ]:
# using a slightly different method than for bfast
bars = gmaps.places('bars near ' + closest_address)

dollar = []
for i in range(0, len(bars['results'])):
	try:
	    dollar.append(bars['results'][i]['price_level'])
	except:
	    dollar.append(0)
fancy_bar = bars['results'][dollar.index(max(dollar))]

print('Fanciest bar is {} at {}'.format(fancy_bar['name'], fancy_bar['formatted_address'])) # fanciest